In [ ]:
# Gemini-as-Judge Evaluation Analysis
# Imports, loads all CSVs, and runs the full analysis pipeline.

import sys
from pathlib import Path

sys.path.insert(0, str(Path().resolve()))

from analyze_gemini_evals import (
    load_all,
    per_model_summary,
    assessment_pivot,
    tuning_delta,
    hardest_questions,
    build_report,
    OUT_DIR,
)
import pandas as pd

OUT_DIR.mkdir(parents=True, exist_ok=True)

frames  = load_all()
summary = per_model_summary(frames)
pivot   = assessment_pivot(summary)
delta   = tuning_delta(frames)
hardest = hardest_questions(frames)

print("Loaded models:", list(frames.keys()))

## 1. Overall Model Ranking
Sorted by mean Gemini score (descending). F1 is excluded.

In [ ]:
ranking_cols = [
    "model", "n_valid", "mean_score", "median_score", "std_score",
    "pct_CORRECT", "pct_PARTIALLY_CORRECT", "pct_INCORRECT", "pct_HALLUCINATED",
]
display(
    summary[ranking_cols]
    .sort_values("mean_score", ascending=False)
    .style
    .format({c: "{:.1f}" for c in ranking_cols if c.startswith("pct_") or c.endswith("_score")})
    .background_gradient(subset=["mean_score"], cmap="RdYlGn")
    .background_gradient(subset=["pct_CORRECT"], cmap="Greens")
    .background_gradient(subset=["pct_HALLUCINATED"], cmap="Reds")
    .set_caption("Model ranking by mean Gemini score")
)

## 2. Assessment Distribution (stacked bar chart)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

ASSESSMENT_ORDER = ["CORRECT", "PARTIALLY_CORRECT", "INCORRECT", "HALLUCINATED"]
COLORS = {"CORRECT": "#2ecc71", "PARTIALLY_CORRECT": "#f39c12",
          "INCORRECT": "#e74c3c", "HALLUCINATED": "#8e44ad"}

plot_df = pivot.set_index("model")
plot_df = plot_df.loc[summary.sort_values("mean_score", ascending=False)["model"]]

fig, ax = plt.subplots(figsize=(10, 5))
bottom = np.zeros(len(plot_df))
for a in ASSESSMENT_ORDER:
    vals = plot_df[a].values
    bars = ax.barh(plot_df.index, vals, left=bottom, color=COLORS[a], label=a)
    for bar, val, b in zip(bars, vals, bottom):
        if val >= 4:
            ax.text(b + val / 2, bar.get_y() + bar.get_height() / 2,
                    f"{val:.1f}%", ha="center", va="center", fontsize=9, color="white", fontweight="bold")
    bottom += vals

ax.set_xlabel("Percentage of questions (%)")
ax.set_title("Gemini Assessment Distribution per Model")
ax.legend(loc="lower right", bbox_to_anchor=(1.22, 0))
ax.set_xlim(0, 100)
plt.tight_layout()
plt.savefig(OUT_DIR / "assessment_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Tuning Impact (Gemma & Llama — paired on shared question IDs)

In [ ]:
for _, row in delta.iterrows():
    fam = row["model_family"]
    print(f"{'='*55}")
    print(f"  {fam}  |  n paired = {int(row['n_paired'])}")
    print(f"{'='*55}")
    print(f"  {'Metric':<28} {'Not tuned':>10} {'Tuned':>10} {'Delta':>8}")
    print(f"  {'-'*28} {'-'*10} {'-'*10} {'-'*8}")
    print(f"  {'Mean score':<28} {row['mean_score_untuned']:>10.1f} {row['mean_score_tuned']:>10.1f} {row['mean_score_delta']:>+8.1f}")
    for a in ASSESSMENT_ORDER:
        u = row[f"pct_{a}_untuned"]
        t = row[f"pct_{a}_tuned"]
        d = row[f"pct_{a}_delta"]
        print(f"  {a+' %':<28} {u:>10.1f} {t:>10.1f} {d:>+8.1f}")
    print()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (_, row) in zip(axes, delta.iterrows()):
    fam    = row["model_family"]
    labels = ASSESSMENT_ORDER
    untuned_vals = [row[f"pct_{a}_untuned"] for a in labels]
    tuned_vals   = [row[f"pct_{a}_tuned"]   for a in labels]
    x = np.arange(len(labels))
    w = 0.35
    ax.bar(x - w/2, untuned_vals, w, label="Not tuned", color=[COLORS[a] for a in labels], alpha=0.55)
    ax.bar(x + w/2, tuned_vals,   w, label="Tuned",     color=[COLORS[a] for a in labels], alpha=1.0)
    ax.set_xticks(x)
    ax.set_xticklabels([a.replace("_", "\n") for a in labels], fontsize=9)
    ax.set_ylabel("% of questions")
    ax.set_title(f"{fam}: tuned vs not tuned")
    ax.legend()
    ax.set_ylim(0, 60)
plt.suptitle("Assessment distribution: tuning impact", fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(OUT_DIR / "tuning_impact.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Hardest Questions (bottom 20 by avg Gemini score across all models)

In [ ]:
display(
    hardest[["id", "question", "avg_score", "n_models",
             "n_correct", "n_partial", "n_incorrect", "n_hallucinated"]]
    .style
    .format({"avg_score": "{:.1f}"})
    .background_gradient(subset=["avg_score"], cmap="RdYlGn")
    .background_gradient(subset=["n_hallucinated"], cmap="Purples")
    .set_caption("20 hardest questions (lowest avg Gemini score)")
)

## 5. Key Observations

In [ ]:
report = build_report(summary, pivot, delta, hardest, frames)
report_path = OUT_DIR / "report.md"
report_path.write_text(report, encoding="utf-8")

from IPython.display import Markdown
display(Markdown(report))